# 02 the contact and the booking form in a browser

Notebook 01 asks the backends what they would answer. This one asks the page
what it actually sends, because everything between the two lives in the bundle:
the query router that opens the dialog, the language the form is read in and
therefore the template it picks, the shape gqty builds, and whether the visitor
is told the truth about what happened.

Chromium, headless, read only. Both backends are intercepted at the network
edge, so nothing reaches emailwerk and nothing reaches the pylon: no mail is
sent and no booking row is written, whatever the form is filled with. The
interception is the safety, and it is the only one. `smoke@example.invalid` in
the email field is not a second line of defence: an invalid reply-to only
bounces the visitor's own confirmation copy, while the request template carries
the company's stored recipients, so a send that escaped would land in the
office's inbox and a booking that escaped would be a real row in dispatch. That
is why the routes below match a host rather than a URL, and why a backstop at
the context level aborts anything naming a mutation that no route claimed.

Because the requests are answered before they leave the browser, the notebook is
equally valid against a `*.pages.dev` deployment, where a real call to emailwerk
would fail CORS. That holds because the intercepted answer carries the origin of
the page it is answering rather than the production domain: Chromium checks a
fulfilled response like any other, and one naming the wrong origin would be
thrown away before the site ever saw it. That is what the workflow uses to find
out whether rolling back would help.

The seven things it proves:

1. the link `?contact=…` opens the dialog with the message it carried;
2. submitting posts exactly one anonymous mutation to emailwerk;
3. that mutation carries the template of the page's own language, the visitor's
   address as `replyTo`, and no recipient of its own;
4. a success closes the dialog;
5. a refusal keeps it open and says so;
6. `?booking` posts `bookTransfer` to this brand's pylon **first** and the mail
   afterwards, carrying the code the pylon minted;
7. a refused booking still sends the enquiry, with an empty code, and says so.

In [ ]:
import json
import pathlib
import queue
import re
import sys
import threading
import time
from datetime import date, timedelta

from playwright.sync_api import sync_playwright

sys.path.insert(0, str(pathlib.Path.cwd()))
sys.path.insert(0, str(pathlib.Path.cwd() / "tests"))
import sitekit as k

k.start_run("02-contact-form-browser")

SITE = k.fetch_site()
DEPLOYED = SITE["templates"]
EMAILWERK = k.CONFIG["emailwerk_url"]
PYLON = (k.pylon_urls_in_bundle(SITE["bundle"]) or [k.PYLON_URL])[0]

# The marker travels through the URL, into the textarea, into the mutation's
# `values.message`, which is how the whole chain is proven with one string.
MARKER = "post-deploy smoke marker"
EMAIL = "smoke@example.invalid"
PHONE = "+43 660 1234567"

PICKUP_DAY = (date.today() + timedelta(days=2)).isoformat()
PICKUP_TIME = "10:30"
PICKUP = "Flughafen Wien, Objekt 1"
DROPOFF = "Hotel Sacher, Wien"
PASSENGERS = "3"

# What the intercepted backends answer. The success body is the shape
# gatsby-jaen-emailwerk selects (`__typename`, `id`, `status`); the refusal is
# the exact error the live templates gave for ten days in September.
MAIL_OK = {"data": {"sendTemplateMail": {"__typename": "Message", "id": "smoke", "status": "QUEUED"}}}
MAIL_REFUSED = {"data": None, "errors": [{
    "message": "Diese Vorlage hat keinen hinterlegten Empfänger.",
    "extensions": {"code": "PUBLIC_SEND_NO_STORED_RECIPIENT"}}]}
BOOKING_OK = {"data": {"bookTransfer": {"__typename": "Transfer",
                                        "id": "transfer:smoke", "code": "SMOKE1-1"}}}
BOOKING_REFUSED = {"data": None, "errors": [{
    "message": "smoke: the pylon refused this booking on purpose",
    "extensions": {"code": "SMOKE_DRILL"}}]}

print("site:", k.CONFIG["site_url"])
print("deployed templates:", json.dumps(DEPLOYED))
print("pylon:", PYLON)

## The browser lives in a thread of its own

Playwright's synchronous API refuses to start inside a running asyncio loop,
and a Jupyter kernel is exactly that, so the browser is given a worker thread
and every step is one call into it. The alternative is the async API and an
`await` on every line, which reads worse and hides where the waiting happens.

It also shapes the notebook usefully: a step in the browser is a function that
does a whole scene and answers with plain data, and every assertion below is
made on that data, outside the browser. Nothing here can accidentally depend on
the page still being in a particular state three cells later.

In [ ]:
class BrowserThread:
    """One thread that owns playwright, and a queue of things to do in it.

    Playwright's sync objects belong to the thread that created them, so both
    the browser and every locator built from it stay behind this queue.
    """

    def __init__(self):
        self._work = queue.Queue()
        self._thread = threading.Thread(target=self._serve, daemon=True)
        self._thread.start()

    def _serve(self):
        while True:
            job, box = self._work.get()
            if job is None:
                box.put((None, None))
                return
            try:
                box.put((job(), None))
            except BaseException as err:  # handed back to the caller intact
                box.put((None, err))

    def run(self, fn, *args, **kwargs):
        box = queue.Queue()
        self._work.put((lambda: fn(*args, **kwargs), box))
        value, err = box.get()
        if err is not None:
            raise err
        return value

    def stop(self):
        box = queue.Queue()
        self._work.put((None, box))
        box.get()


WORKER = BrowserThread()
# Everything the browser thread owns. Only functions running inside it may
# touch this.
STATE = {"pages": {}}


def in_browser(fn, *args, **kwargs):
    return WORKER.run(fn, *args, **kwargs)


def scenario(fn, *args, **kwargs):
    """One scene in the browser, as data. A failure becomes a record too, so
    the checks below report it instead of the cell dying without evidence."""
    try:
        return in_browser(fn, *args, **kwargs)
    except Exception as err:
        return {"error": f"{type(err).__name__}: {err}"}

## The handles the page offers

Every selector below comes from the live DOM, read on 2026-09-17, and none of
them is a label: the site speaks four languages and the same form is German,
English, Turkish or Arabic depending on which root it was opened from.

What the DOM offers instead is stable and language independent. react-hook-form
puts the registered name on the element, so `[name=firstName]` and its siblings
are the fields. The consent control is a Chakra v3 checkbox whose real `<input>`
is clipped to a single pixel, so the thing to click is its
`label[data-scope=checkbox][data-part=root]`. The submit is the only
`button[type=submit]` in the dialog. A toast carries `data-type` as `success`,
`error` or `warning`, which is how a refusal is recognised without reading a
translated sentence.

The cookie banner is jaen's `#cc--main`. It renders under the dialog rather
than over it, so it never blocks the form, but it is dismissed anyway: accepted
when it can be clicked, hidden when the dialog's own overlay owns the pointer.

In [ ]:
TOASTS = ("() => Array.from(document.querySelectorAll('[data-scope=toast][data-part=root]'))"
          ".map(e => e.getAttribute('data-type'))")


# Anything that named a mutation and was not claimed by a route below. It has to
# stay empty, and the last check of the notebook says so.
ESCAPED = []


def guard_the_network(context):
    """Nothing that would send or write may leave this browser.

    The two routes below claim the two calls the forms make, and they claim them
    by host, which is a thing the site can change. This sits behind them, at the
    context level, so the page's own routes are consulted first, and refuses any
    POST naming a mutation that nobody answered. A host that moved then reads as
    a form that failed, which the checks already report, instead of a real
    enquiry in the office's inbox and a real row in dispatch, four times a day
    on the schedule.
    """
    def handler(route):
        body = route.request.post_data or ""
        if "sendTemplateMail" in body or "bookTransfer" in body:
            ESCAPED.append(route.request.url)
            route.abort()
        else:
            route.continue_()
    context.route("**/*", handler)


def start_browser():
    STATE["pw"] = sync_playwright().start()
    STATE["browser"] = STATE["pw"].chromium.launch(headless=True)
    STATE["context"] = STATE["browser"].new_context(viewport={"width": 1440, "height": 1400})
    guard_the_network(STATE["context"])
    return STATE["browser"].version


def stop_browser():
    for page in STATE["pages"].values():
        try:
            page.close()
        except Exception:
            pass
    STATE["context"].close()
    STATE["browser"].close()
    STATE["pw"].stop()
    return "closed"


def dismiss_cookie_banner(page):
    if not page.locator("#cc--main").count():
        return "no banner"
    try:
        page.locator("#c-p-bn").first.click(timeout=2000)
        return "accepted"
    except Exception:
        page.evaluate("() => { const n = document.querySelector('#cc--main');"
                      " if (n) n.style.display = 'none' }")
        return "hidden"


def capture_mail(page, payload, into):
    def handler(route):
        request = route.request
        body = request.post_data or ""
        if "sendTemplateMail" not in body:
            # The whole host is claimed, so only the mutation is answered and
            # nothing else about emailwerk is faked.
            route.continue_()
            return
        into.append({"what": "emailwerk", "method": request.method,
                     "headers": {a.lower(): b for a, b in request.headers.items()},
                     "body": body, "at": time.time()})
        route.fulfill(status=200, headers={
            "content-type": "application/json",
            # Chromium checks a fulfilled response like any other, so the
            # allowance has to name the origin that asked rather than
            # SITE_ORIGIN, which stays the production domain on purpose. Against
            # a `*.pages.dev` deployment those differ, and every answer would be
            # thrown away by the browser before the form ever saw it.
            "access-control-allow-origin": request.headers.get("origin") or k.browser_origin(),
            "access-control-allow-credentials": "true",
        }, body=json.dumps(payload))
    page.route(k.on_host(EMAILWERK), handler)


def capture_booking(page, payload, into):
    def handler(route):
        request = route.request
        body = request.post_data or ""
        if "bookTransfer" not in body:
            # The form also reads the fleet from this host; only the booking is
            # ours to answer.
            route.continue_()
            return
        into.append({"what": "pylon", "method": request.method,
                     "headers": {a.lower(): b for a, b in request.headers.items()},
                     "body": body, "at": time.time()})
        route.fulfill(status=200, headers={
            "content-type": "application/json",
            "access-control-allow-origin": request.headers.get("origin") or k.browser_origin(),
        }, body=json.dumps(payload))
    page.route(k.on_host(PYLON), handler)


def open_form(name, query):
    page = STATE["context"].new_page()
    STATE["pages"][name] = page
    page.goto(k.CONFIG["site_url"] + query, wait_until="networkidle", timeout=90_000)
    page.wait_for_timeout(2500)
    dismiss_cookie_banner(page)
    return page


def dialog_of(page, field):
    """The modal that holds a given field, which is never the cookie banner."""
    return page.locator('[role="dialog"]').filter(has=page.locator(f'[name="{field}"]'))


def submit_and_watch(page, dialog, seconds=14.0):
    """Press send and watch what the visitor is shown.

    A toast lives five to seven seconds and then removes itself, so the DOM is
    polled rather than read once: a single look afterwards would miss the
    success toast of a booking that also warned.
    """
    dialog.locator('button[type="submit"]').click()
    seen, closed_after = [], None
    started = time.time()
    while time.time() - started < seconds:
        for kind in page.evaluate(TOASTS):
            if kind and kind not in seen:
                seen.append(kind)
        if closed_after is None and dialog.count() == 0:
            closed_after = round(time.time() - started, 1)
        if closed_after is not None and seen and time.time() - started > 4:
            break
        page.wait_for_timeout(250)
    return {"toasts": seen, "closed_after": closed_after}


def save_evidence(name):
    """A screenshot and the page as it stands, for a failure nobody watched."""
    page = STATE["pages"].get(name)
    if page is None:
        return "no page to photograph"
    out = k.out_dir()
    shot, markup = out / f"{name}.png", out / f"{name}.html"
    try:
        page.screenshot(path=str(shot), full_page=True)
    except Exception as err:
        shot = f"(no screenshot: {err})"
    try:
        markup.write_text(page.content(), encoding="utf-8")
    except Exception as err:
        markup = f"(no html: {err})"
    return f"{shot}, {markup}"


def keep(c, name):
    if c.status == "FAIL":
        c.note("evidence: " + str(in_browser(save_evidence, name)))


def args_of(record):
    """The argument object of a captured mutation.

    gqty names its variables with a hash of the selection, so the arguments
    cannot be addressed by path; the body is walked for the key instead.
    """
    return k.find_dict_with(json.loads(record["body"]), "templateId") or {}


print("chromium", in_browser(start_browser))

## 1. The link opens the dialog, and the page's language decides the template

`?contact=<text>` is how every "write to us" link on the site opens the form,
with the enquiry already typed in. `useQueryRouter` reads the parameter and
`ContactModal` puts it into the textarea, so a dialog without the marker means
the link is broken even though the page looks right.

The language is read off `<html lang>` rather than assumed. `templateForLocale`
takes the first two letters of the tag and falls back to German, so the page
itself decides which of the four templates the form must send, and this
notebook has to predict the same one before the form is submitted.

In [ ]:
def contact_open():
    page = open_form("contact", "/?contact=" + MARKER.replace(" ", "%20"))
    dialog = dialog_of(page, "message")
    return {
        "dialogs": dialog.count(),
        "prefilled": dialog.locator('[name="message"]').input_value() if dialog.count() else None,
        "lang": page.evaluate("document.documentElement.lang"),
    }


CONTACT = scenario(contact_open)
EXPECTED_ID = k.template_for_lang(CONTACT.get("lang"), DEPLOYED)

with k.check("the contact dialog opens with the message the link carried") as c:
    if CONTACT.get("error"):
        c.fail(CONTACT["error"])
    else:
        c.expect(CONTACT["dialogs"] == 1, f"exactly one contact dialog (got {CONTACT['dialogs']})")
        c.expect(CONTACT["prefilled"] == MARKER,
                 f"the textarea carries the marker (got {CONTACT['prefilled']!r})")
    keep(c, "contact")

with k.check("the page's language names the template the form has to send") as c:
    c.note(f"<html lang> is {CONTACT.get('lang')!r}")
    c.expect(bool(CONTACT.get("lang")), "the page declares a language")
    c.expect(bool(EXPECTED_ID), f"the deployed bundle has a template for {CONTACT.get('lang')!r}")
    c.note(f"expecting templateId {EXPECTED_ID}")

## 2. Submitting posts one anonymous mutation, and nothing leaves the browser

The route was installed before the page was opened and answers the body
emailwerk answers on success. Filling the form and pressing send is therefore a
complete exercise of the site's own code path, with the network cut at the last
possible moment.

In [ ]:
def contact_submit():
    page = STATE["pages"]["contact"]
    captured = []
    capture_mail(page, MAIL_OK, captured)
    dialog = dialog_of(page, "message")
    dialog.locator('[name="firstName"]').fill("Post")
    dialog.locator('[name="lastName"]').fill("Deploy")
    dialog.locator('[name="email"]').fill(EMAIL)
    dialog.locator('[name="phone"]').fill(PHONE)
    dialog.locator('label[data-scope="checkbox"][data-part="root"]').first.click()
    shown = submit_and_watch(page, dialog)
    return {"captured": captured, **shown}


SENT = scenario(contact_submit)
MAIL_SENT = SENT.get("captured") or []

with k.check("the form posts exactly one request to the mail gateway") as c:
    if SENT.get("error"):
        c.fail(SENT["error"])
    c.expect(len(MAIL_SENT) == 1, f"one POST to {EMAILWERK} (got {len(MAIL_SENT)})")
    if MAIL_SENT:
        c.expect(MAIL_SENT[0]["method"] == "POST", "it is a POST")
        c.expect("application/json" in MAIL_SENT[0]["headers"].get("content-type", ""),
                 "it is JSON")
    keep(c, "contact")

## 3. What that mutation carries

This is the contract of notebook 01 seen from the other side. Four things have
to hold and each of them has been wrong somewhere in this stack before.

The document selects `sendTemplateMail` and nothing else, so no second root
field rides along; gqty aliases everything it selects, so the selection set is
parsed rather than searched for a substring. The template is the one for the
page's language, because a German visitor answered in English is the defect
that made these four templates exist. The envelope override carries only the
visitor's address as `replyTo`: emailwerk refuses a `to` or a `subject` from an
anonymous caller outright, and that refusal would be a form that simply stops
working. And the request carries no `Authorization`, because a visitor has no
session and the anonymous branch is the one under test.

In [ ]:
with k.check("the document selects sendTemplateMail and nothing else") as c:
    if not MAIL_SENT:
        c.fail("nothing was captured")
    else:
        document = json.loads(MAIL_SENT[0]["body"]).get("query", "")
        fields = k.root_fields(document)
        c.expect(fields == ["sendTemplateMail"],
                 f"the only root field is sendTemplateMail (got {fields})")

with k.check("the template is the one for the language the page is read in") as c:
    if not MAIL_SENT:
        c.fail("nothing was captured")
    else:
        args = args_of(MAIL_SENT[0])
        c.expect(args.get("templateId") == EXPECTED_ID,
                 f"templateId is {EXPECTED_ID} (got {args.get('templateId')})")
        c.expect("to" not in args, "the site chooses no recipient of its own")
        c.expect("scheduledAt" not in args, "and asks for no scheduling")

with k.check("the envelope override is the visitor's address and nothing more") as c:
    if not MAIL_SENT:
        c.fail("nothing was captured")
    else:
        override = args_of(MAIL_SENT[0]).get("envelopeOverride") or {}
        c.expect(override.get("replyTo") == EMAIL,
                 f"replyTo is what was typed (got {override.get('replyTo')!r})")
        c.expect(override.get("subject") is None, "no subject is overridden")
        c.expect(override.get("to") is None, "no recipient is overridden")

with k.check("the values carry the enquiry, including the message from the link") as c:
    if not MAIL_SENT:
        c.fail("nothing was captured")
    else:
        values = args_of(MAIL_SENT[0]).get("values") or {}
        for field, expected in (("firstName", "Post"), ("lastName", "Deploy"),
                                ("email", EMAIL), ("message", MARKER)):
            c.expect(values.get(field) == expected,
                     f"{field} is {expected!r} (got {values.get(field)!r})")
        c.expect(bool(values.get("invokedOnUrl")),
                 f"invokedOnUrl says where it came from ({values.get('invokedOnUrl')!r})")

with k.check("the request is anonymous, as a visitor's is") as c:
    if not MAIL_SENT:
        c.fail("nothing was captured")
    else:
        c.expect("authorization" not in MAIL_SENT[0]["headers"],
                 "no Authorization header is attached")

## 4. A success is shown as a success

`onSubmit` closes the dialog only when the client reports `ok`. That flag, and
not the presence of `errors`, is what the site reads, because a transport
failure leaves `errors` undefined and the visitor used to be congratulated for
a mail that never went anywhere.

In [ ]:
with k.check("the dialog closes and nothing red is shown") as c:
    c.note(f"toasts seen: {SENT.get('toasts')}, dialog closed after {SENT.get('closed_after')}s")
    c.expect(SENT.get("closed_after") is not None, "the dialog is gone within the watch window")
    c.expect("error" not in (SENT.get("toasts") or []), "no error toast appeared")
    c.expect("success" in (SENT.get("toasts") or []), "the visitor is told it was sent")
    keep(c, "contact")

## 5. A refusal is shown as a refusal

The same form, answered with the exact error the live templates gave once their
recipients were wiped. The visitor must keep the dialog, with everything they
typed still in it, and must be told that it failed. A dialog that closes here
is the worst outcome of all: the enquiry is gone and the visitor believes it
arrived.

In [ ]:
def contact_refused():
    captured = []
    page = open_form("contact-refused", "/?contact=" + MARKER.replace(" ", "%20"))
    capture_mail(page, MAIL_REFUSED, captured)
    dialog = dialog_of(page, "message")
    dialog.locator('[name="firstName"]').fill("Post")
    dialog.locator('[name="lastName"]').fill("Deploy")
    dialog.locator('[name="email"]').fill(EMAIL)
    dialog.locator('[name="phone"]').fill(PHONE)
    dialog.locator('label[data-scope="checkbox"][data-part="root"]').first.click()
    shown = submit_and_watch(page, dialog, seconds=10.0)
    return {"captured": captured, **shown}


REFUSED = scenario(contact_refused)

with k.check("a refused send keeps the dialog and says so") as c:
    if REFUSED.get("error"):
        c.fail(REFUSED["error"])
    c.note(f"toasts seen: {REFUSED.get('toasts')}")
    c.expect(len(REFUSED.get("captured") or []) == 1,
             f"the mutation was attempted once (got {len(REFUSED.get('captured') or [])})")
    c.expect(REFUSED.get("closed_after") is None,
             "the dialog is still open, with what the visitor typed")
    c.expect("error" in (REFUSED.get("toasts") or []), "an error toast appeared")
    c.expect("success" not in (REFUSED.get("toasts") or []),
             "and no success toast appeared beside it")
    keep(c, "contact-refused")

## 6. The booking form, both calls, nothing written and nothing sent

A booking is two calls in a fixed order, and the order is the point. The form
posts `bookTransfer` to this brand's own pylon first, waits for the code it
mints, and only then sends the enquiry mail carrying that code, because the
mail a person reads names the booking. Sending first would produce a mail with
an empty booking number every single time.

Both backends are intercepted. The pylon answers the shape it really answers, a
`Transfer` with an `id` and a `code`, and the code is `SMOKE1-1`, so the two
values the site derives from it can be checked: `bookingCode` is the stem both
legs share and `code` is the outbound leg.

The vehicle class and the model are read out of the dropdowns rather than
typed, because their labels are translated and their contents come from the
backend's fleet whenever it has one. The pickup is two days out: the form
refuses anything less than fifteen minutes from now, the same lead time the
pylon enforces, so a visitor reads a sentence under the field instead of
watching the booking fail.

In [ ]:
def fill_booking(dialog):
    dialog.locator('[name="rideCategory"]').select_option("DISTANCE")
    dialog.locator('[name="rideType"]').select_option("ONEWAY")
    dialog.locator('[name="date"]').fill(PICKUP_DAY)
    dialog.locator('[name="time"]').fill(PICKUP_TIME)
    dialog.locator('[name="pickupAddress"]').fill(PICKUP)
    dialog.locator('[name="destinationAddress"]').fill(DROPOFF)
    dialog.locator('[name="passengers"]').fill(PASSENGERS)
    classes = [v for v in dialog.locator('[name="carClass"]').evaluate(
        "e => Array.from(e.options).map(o => o.value)") if v]
    if classes:
        dialog.locator('[name="carClass"]').select_option(classes[0])
        dialog.page.wait_for_timeout(600)
    models = [v for v in dialog.locator('[name="carTitle"]').evaluate(
        "e => Array.from(e.options).map(o => o.value)") if v]
    if models:
        dialog.locator('[name="carTitle"]').select_option(models[0])
    dialog.locator('[name="paymentOption"]').select_option("CASH")
    dialog.locator('[name="firstName"]').fill("Post")
    dialog.locator('[name="lastName"]').fill("Deploy")
    dialog.locator('[name="email"]').fill(EMAIL)
    dialog.locator('[name="phone"]').fill(PHONE)
    dialog.locator('[name="message"]').fill(MARKER)
    dialog.locator('label[data-scope="checkbox"][data-part="root"]').first.click()
    return {"classes": classes, "models": models}


def book(name, pylon_answer):
    captured = []
    page = open_form(name, "/?booking")
    capture_booking(page, pylon_answer, captured)
    capture_mail(page, MAIL_OK, captured)
    dialog = dialog_of(page, "pickupAddress")
    chosen = fill_booking(dialog)
    shown = submit_and_watch(page, dialog)
    return {"captured": captured, "chosen": chosen, **shown}


BOOKED = scenario(book, "booking", BOOKING_OK)
PYLON_CALLS = [r for r in (BOOKED.get("captured") or []) if r["what"] == "pylon"]
BOOKING_MAIL = [r for r in (BOOKED.get("captured") or []) if r["what"] == "emailwerk"]

with k.check("the booking reaches this brand's pylon before the mail gateway") as c:
    if BOOKED.get("error"):
        c.fail(BOOKED["error"])
    c.note(f"captured: {[r['what'] for r in (BOOKED.get('captured') or [])]}, "
           f"class {(BOOKED.get('chosen') or {}).get('classes', [])[:1]}, "
           f"model {(BOOKED.get('chosen') or {}).get('models', [])[:1]}")
    c.expect(len(PYLON_CALLS) == 1, f"one bookTransfer to {PYLON} (got {len(PYLON_CALLS)})")
    c.expect(len(BOOKING_MAIL) == 1, f"one mail (got {len(BOOKING_MAIL)})")
    if PYLON_CALLS and BOOKING_MAIL:
        c.expect(PYLON_CALLS[0]["at"] < BOOKING_MAIL[0]["at"],
                 "the mail follows the pylon's answer, so it can carry the code")
    keep(c, "booking")

with k.check("the booking document is the literal the site builds") as c:
    if not PYLON_CALLS:
        c.fail("no booking was captured")
    else:
        document = json.loads(PYLON_CALLS[0]["body"]).get("query", "")
        c.expect(PYLON_CALLS[0]["method"] == "POST", "it is a POST")
        c.expect(k.root_fields(document) == ["bookTransfer"],
                 f"the only root field is bookTransfer (got {k.root_fields(document)})")
        c.expect(bool(re.search(r"bookTransfer\(args:\s*\{", document)),
                 "the arguments are one `args` object, written as a literal")
        for field in ("pickupDateTime", "pickupLocation", "dropoffLocation",
                      "subject", "paymentMethode", "passengers", "details"):
            c.expect(f"{field}:" in document, f"it carries {field}")
        c.expect("authorization" not in PYLON_CALLS[0]["headers"],
                 "a booking is made without a session")

with k.check("the mail carries the ride and the code the pylon minted") as c:
    if not BOOKING_MAIL:
        c.fail("no mail was captured")
    else:
        args = args_of(BOOKING_MAIL[0])
        values = args.get("values") or {}
        c.expect(args.get("templateId") == EXPECTED_ID,
                 f"templateId is {EXPECTED_ID} (got {args.get('templateId')})")
        c.expect(values.get("bookingCode") == "SMOKE1",
                 f"bookingCode is the stem both legs share (got {values.get('bookingCode')!r})")
        c.expect(values.get("code") == "SMOKE1-1",
                 f"code is the outbound leg (got {values.get('code')!r})")
        for field, expected in (("pickupAddress", PICKUP), ("destinationAddress", DROPOFF),
                                ("date", PICKUP_DAY), ("time", PICKUP_TIME)):
            c.expect(values.get(field) == expected,
                     f"{field} is what was typed (got {values.get(field)!r})")
        c.expect(str(values.get("passengers")) == PASSENGERS,
                 f"passengers is what was typed (got {values.get('passengers')!r})")
        c.expect((args.get("envelopeOverride") or {}).get("replyTo") == EMAIL,
                 "replyTo is the visitor's address")
        c.expect("to" not in args, "no recipient is chosen by the site")

with k.check("the booking dialog closes and nothing red is shown") as c:
    c.note(f"toasts seen: {BOOKED.get('toasts')}, dialog closed after "
           f"{BOOKED.get('closed_after')}s")
    c.expect(BOOKED.get("closed_after") is not None, "the dialog is gone")
    c.expect("error" not in (BOOKED.get("toasts") or []), "no error toast appeared")
    c.expect("success" in (BOOKED.get("toasts") or []), "the visitor is told it was sent")
    keep(c, "booking")

## 7. A refused booking still sends the enquiry, and says the booking is not in

This is the site's design rather than an accident, so it is asserted as it is
written and not as it might be nicer: when the pylon refuses, `booking.tsx`
shows an orange "Booking not yet in the system", sends the mail anyway with
`bookingCode`, `code` and `returnCode` empty, and the template says so. The
office gets the enquiry and completes it by hand.

The failure this guards against is the opposite one, a refused booking that
also swallows the mail, which would leave the visitor with a green toast and
nobody with a ride.

In [ ]:
REFUSED_BOOKING = scenario(book, "booking-refused", BOOKING_REFUSED)
REFUSED_MAIL = [r for r in (REFUSED_BOOKING.get("captured") or []) if r["what"] == "emailwerk"]

with k.check("a refused booking still sends the enquiry, without a code") as c:
    if REFUSED_BOOKING.get("error"):
        c.fail(REFUSED_BOOKING["error"])
    c.note(f"captured: {[r['what'] for r in (REFUSED_BOOKING.get('captured') or [])]}, "
           f"toasts seen: {REFUSED_BOOKING.get('toasts')}")
    c.expect(len(REFUSED_MAIL) == 1, f"the mail still goes out (got {len(REFUSED_MAIL)})")
    if REFUSED_MAIL:
        values = args_of(REFUSED_MAIL[0]).get("values") or {}
        c.expect(values.get("bookingCode") == "",
                 f"bookingCode is empty (got {values.get('bookingCode')!r})")
        c.expect(values.get("code") == "", f"code is empty (got {values.get('code')!r})")
    c.expect("warning" in (REFUSED_BOOKING.get("toasts") or []),
             "the visitor is warned that the booking is not in the system")
    c.expect("success" in (REFUSED_BOOKING.get("toasts") or []),
             "and told that the enquiry was sent")
    c.expect("error" not in (REFUSED_BOOKING.get("toasts") or []),
             "the mail itself did not fail")
    keep(c, "booking-refused")

## 8. The browser is closed, and the verdict

A failing check above has already written its screenshot and the page as it
stood into `tests/out/`, which the workflow uploads as the run's artifact. That
directory is gitignored and is emptied after a local run.
The last check is the backstop's own report. It is empty on every healthy run,
and the run it is not empty on is the one where an endpoint moved and a route
stopped matching, which is the only way this notebook could ever put mail in the
company's inbox.

In [ ]:
with k.check("nothing the forms send ever left the browser") as c:
    c.expect(not ESCAPED, f"no mutation reached a real backend (escaped: {ESCAPED})")

print(in_browser(stop_browser))
WORKER.stop()

k.finish()